In [1]:
from MotorInversion import MotorInversion
import pandas as pd
import importlib
import itertools
from datetime import datetime, date
import matplotlib.pyplot as plt

from UniversoActivos import UniversoActivosEstatico, UniversoActivosDinamico
from ProveedorDatos import YFinanceProvider
from VariablesTransformation import FeatureEngineer
import Modelos
Modelos = importlib.reload(Modelos)
from Estrategia import EstrategiaMLEquiponderada, EstrategiaMLMinVarAlphaTilt
from Backtest import BacktestEngine

RandomForestModel = Modelos.RandomForestModel
XGBoostModel = Modelos.XGBoostModel

In [2]:
import pandas as pd
import requests

def get_eurostoxx50_tickers():
    url = 'https://en.wikipedia.org/wiki/EURO_STOXX_50'
    
    # Añadimos una cabecera para simular un navegador real
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36"
    }
    
    # Hacemos la petición con requests
    response = requests.get(url, headers=headers)
    
    # Pasamos el contenido HTML (response.text) a pandas
    tables = pd.read_html(response.text)
    
    # Buscamos la tabla que contiene la columna 'Ticker'
    df = next(table for table in tables if 'Ticker' in table.columns)
    
    return df['Ticker'].tolist()

tickers = get_eurostoxx50_tickers()
print(f"Total empresas: {len(tickers)}")
print(f"Muestra: {tickers}")

Total empresas: 50
Muestra: ['ADS.DE', 'ADYEN.AS', 'AD.AS', 'AI.PA', 'AIR.PA', 'ALV.DE', 'ABI.BR', 'ARGX.BR', 'ASML.AS', 'CS.PA', 'BAS.DE', 'BAYN.DE', 'BBVA.MC', 'SAN.MC', 'BMW.DE', 'BNP.PA', 'BN.PA', 'DBK.DE', 'DB1.DE', 'DHL.DE', 'DTE.DE', 'ENEL.MI', 'ENI.MI', 'EL.PA', 'RACE.MI', 'RMS.PA', 'IBE.MC', 'ITX.MC', 'IFX.DE', 'INGA.AS', 'ISP.MI', 'OR.PA', 'MC.PA', 'MBG.DE', 'MUV2.DE', 'NDA-FI.HE', 'PRX.AS', 'RHM.DE', 'SAF.PA', 'SGO.PA', 'SAN.PA', 'SAP.DE', 'SU.PA', 'SIE.DE', 'ENR.DE', 'TTE.PA', 'DG.PA', 'UCG.MI', 'VOW.DE', 'WKL.AS']


C:\Users\jpuerta\AppData\Local\Temp\ipykernel_27908\1949759624.py:16: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  tables = pd.read_html(response.text)


In [3]:
universo   = UniversoActivosEstatico(tickers)
fe         = FeatureEngineer(criterio=5, ticker_indice="^STOXX50E")
modelo     = modelo = RandomForestModel(
    n_estimators=250,
    max_depth=4,
    class_weight=None,
    random_state=528,
    positive_class_weight=10.0
)
estrategia = EstrategiaMLEquiponderada(
    modelo=modelo,
    n_activos_obj=15,
    umbral_salida=22
)

motor = MotorInversion(
    universo       = universo,
    feature_engineer = fe,
    estrategia     = estrategia,
    proveedor_cls  = YFinanceProvider,
    capital_total  = 10000000,
    estado_path    = "./mi_cartera",   # carpeta donde guarda los CSV
    len_ventana = 4
)

In [4]:
# Código para reentrenar el modelo por si cambiamos cosas en el codigo
motor._ultimo_train = None
motor._reentrenar_si_toca(date(2026, 3, 4))  # O la fecha que quieras
motor._guardar_modelo()

c:\Users\jpuerta\OneDrive - Analistas Financieros Internacionales (Afi)\Documentos\Master Quant\Gestión de activos\Trabajo gestión cuantitativa\Trabajo_gestion_cuantitativa\ProveedorDatos.py:72: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly = weekly.agg(


In [ ]:
# Cada viernes ejecutas esto:
señales = motor.ejecutar()

[*********************100%***********************]  51 of 51 completed
[*********************100%***********************]  51 of 51 completed


Compras hechas a fecha 2026-03-13 00:00:00
  Ticker Accion  Cantidad      Precio     CT  Precio_Ejecutado
 ASML.AS COMPRA       564 1180.400024 1.1804         1181.5804
  ENR.DE COMPRA      4362  152.649994 0.1526          152.8026
 BBVA.MC COMPRA     36623   18.184999 0.0182           18.2032
  ADS.DE COMPRA      4716  141.199997 0.1412          141.3412
  PRX.AS COMPRA     14659   45.430000 0.0454           45.4754
  SAP.DE COMPRA      3987  167.020004 0.1670          167.1870
   EL.PA COMPRA      3159  210.800003 0.2108          211.0108
 RACE.MI COMPRA      2276  292.600006 0.2926          292.8926
  WKL.AS COMPRA      9919   67.139999 0.0671           67.2071
 BAYN.DE COMPRA     17020   39.130001 0.0391           39.1691
ADYEN.AS COMPRA       726  917.299988 0.9173          918.2173
  DBK.DE COMPRA     25909   25.705000 0.0257           25.7307
  RHM.DE COMPRA       429 1550.500000 1.5505         1552.0505
 ARGX.BR COMPRA      1081  616.000000 0.6160          616.6160
  AIR.PA COM

Viernes 20 de marzo de 2026

In [5]:
fecha = date(2026, 3, 20)
señales = motor.ejecutar(fecha)

c:\Users\jpuerta\OneDrive - Analistas Financieros Internacionales (Afi)\Documentos\Master Quant\Gestión de activos\Trabajo gestión cuantitativa\Trabajo_gestion_cuantitativa\ProveedorDatos.py:72: FutureWarning: DataFrameGroupBy.resample operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  weekly = weekly.agg(


  Ticker Accion  Cantidad      Precio       CT  Precio_Ejecutado
   EL.PA COMPRA        74  194.750000 0.097375        194.847375
ADYEN.AS COMPRA         3  863.500000 0.431750        863.931750
  PRX.AS COMPRA      1077   40.020000 0.020010         40.040010
  ENR.DE COMPRA       112  140.750000 0.070375        140.820375
  ADS.DE COMPRA         1  133.500000 0.066750        133.566750
 RACE.MI COMPRA        24  273.799988 0.136900        273.936888
  SAP.DE COMPRA       107  153.820007 0.076910        153.896917
  RMS.PA COMPRA       380 1656.000000 0.828000       1656.828000
  ENI.MI COMPRA     26650   23.620001 0.011810         23.631811
  SGO.PA COMPRA      9246   68.080002 0.034040         68.114042
  WKL.AS  VENTA       294   65.440002 0.032720         65.407282
 ARGX.BR  VENTA         4  584.799988 0.292400        584.507588
 BAYN.DE  VENTA       612   38.384998 0.019192         38.365806
  DBK.DE  VENTA       472   24.760000 0.012380         24.747620
 BBVA.MC  VENTA     36623

In [18]:
import yfinance as yf

# Definir el ticker (ejemplo: Apple)
ticker = "ONSW.DE"

# Descargar datos con periodo de 1 día e intervalo de 1 minuto
data = yf.download(ticker, period="5d", interval="1d")

# Mostrar las últimas filas
print(data.tail())

[*********************100%***********************]  1 of 1 completed

Price        Close    High     Low    Open  Volume
Ticker     ONSW.DE ONSW.DE ONSW.DE ONSW.DE ONSW.DE
Date                                              
2026-03-16  5.0304  5.0307  5.0304  5.0307      24
2026-03-17  5.0307  5.0307  5.0303  5.0307      17
2026-03-18  5.0310  5.0310  5.0310  5.0310       0
2026-03-19  5.0317  5.0319  5.0314  5.0319    3998
2026-03-20  5.0320  5.0323  5.0320  5.0323    3998
